In [ ]:
import numpy as np
import pandas as pd
import shutil, time, os, requests, random, copy
import torch
from torchvision import datasets, transforms, models
import torch.nn as nn
import torch.nn.functional as tF
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torch.optim.optimizer import Optimizer, required
import re
from PIL import Image
import os
from sklearn.manifold import TSNE
import glob
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, roc_auc_score, roc_curve, f1_score, precision_score, recall_score
from sklearn.cluster import KMeans
from torch.utils.data import Subset
import torch.optim as optim
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from tabulate import tabulate
from tiatoolbox.tools.stainnorm import MacenkoNormalizer  # add stain normalizationnnnnnnnnnnnnnnnnnn
from sklearn.svm import OneClassSVM


In [ ]:
import warnings
warnings.filterwarnings("ignore")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
###########################################################################################
def set_seed(seed=30):
    np.random.seed(seed)
    torch.manual_seed(seed)
    random.seed(seed)
set_seed(30)
############################################################
NUM_WORKERS = 2

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
#############################################################
train_images_BC_tumor= '/content/drive/MyDrive/demo_2025_pathology/train'
#############################################################
# combine datasets
train_images = '/content/drive/MyDrive/demo_2025_pathology/train'
validation_images = '/content/drive/MyDrive/demo_2025_pathology/valid'
test_images = '/content/drive/MyDrive/demo_2025_pathology/test'

##############################################################

stain_normalizer_BC = MacenkoNormalizer()  # select MacenkoNormalizer6
image_path = '/content/drive/MyDrive/demo_2025_pathology/BC_tissue1_LumA_TCGA-GM-A2DL-01Z-00-DX1.1CE6992E-B1CE-45AB-B2EB-75338B6FEE9D [d=2.02593,x=38379,y=52382,w=519,h=519].png'
target_img_database_BC = Image.open(image_path).convert("RGB")
stain_normalizer_BC.fit(np.array(target_img_database_BC))

In [ ]:
def calculate_mean_std(dataloader):
    mean = torch.zeros(3)
    std = torch.zeros(3)
    total_images_count = 0
    for images, _ in dataloader:
        batch_samples = images.size(0)
        images = images.view(batch_samples, images.size(1), -1)
        mean += images.mean(2).sum(0)
        std += images.std(2).sum(0)
        total_images_count += batch_samples
    mean /= total_images_count
    std /= total_images_count
    return mean, std

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, image_dir, stain_normalizer, transform=None):
        self.image_dir = image_dir
        self.transform = transform
        self.stain_normalizer = stain_normalizer  # for stainnormalizationnnnn
        self.images = (glob.glob(os.path.join(self.image_dir, "*.tif")) + glob.glob(
            os.path.join(self.image_dir, "*.jpg")) + glob.glob(os.path.join(self.image_dir, "*.png")))
    def __len__(self):
        return len(self.images)
    def __getitem__(self, idx):
        img_path = self.images[idx]
        image = Image.open(img_path).convert("RGB")
        image = np.array(image)
        try:
            normalized_image = self.stain_normalizer.transform(image)
        except ValueError as e:
            #print(f"[Warning] Skipping image due to empty tissue mask: {self.images[idx]}")
            return self.__getitem__((idx + 1) % len(self.images))  # Try next image safely

        if self.transform:
            normalized_image = Image.fromarray(normalized_image)
            normalized_image = self.transform(normalized_image)
        image_name = os.path.splitext(img_path)[0].split('/')[-1]
        tissue_number = re.search(r'tissue(\d)', image_name)
        real_label = int(tissue_number.group(1))
        return normalized_image, real_label

In [ ]:
dataset_train_BC = CustomDataset(train_images_BC_tumor, stain_normalizer_BC,transform=transforms.Compose( [transforms.Resize((50, 50)), transforms.ToTensor()]), )
train_dataloader_for_stats_just_tumor = DataLoader(dataset_train_BC, batch_size=8, shuffle=False,num_workers=NUM_WORKERS)
mean_BC, std_BC = calculate_mean_std(train_dataloader_for_stats_just_tumor)
#####
mean = (mean_BC)
std = (std_BC)
model_transforms = transforms.Compose([transforms.Resize((50, 50)), transforms.ToTensor(),transforms.Normalize(mean=mean.tolist(), std=std.tolist())])
# Dictionary for stain normalizers based on dataset directory
stain_normalizers = {train_images_BC_tumor: stain_normalizer_BC}

In [ ]:
def create_dataloaders(train_dir: str, validation_dir: str, test_dir: str, transform: transforms.Compose,
                       batch_size: int, num_workers: int = NUM_WORKERS):
    # Select appropriate stain normalizer based on directory
    stain_normalizer_train = stain_normalizers.get(train_dir, None)
    stain_normalizer_validation = stain_normalizers.get(validation_dir, None)
    stain_normalizer_test = stain_normalizers.get(test_dir, None)

    dataset_train = CustomDataset(train_dir, stain_normalizer=stain_normalizer_train, transform=transform)
    dataset_validation = CustomDataset(validation_dir, stain_normalizer=stain_normalizer_validation,transform=transform)
    dataset_test = CustomDataset(test_dir, stain_normalizer=stain_normalizer_test, transform=transform)
    train_dataloader = DataLoader(dataset_train, batch_size=batch_size, shuffle=True, num_workers=num_workers,pin_memory=False)
    validation_dataloader = DataLoader(dataset_validation, batch_size=batch_size, shuffle=True, num_workers=num_workers,pin_memory=False)
    test_dataloader = DataLoader(dataset_test, batch_size=batch_size, shuffle=True, num_workers=num_workers,pin_memory=False)
    return train_dataloader, validation_dataloader, test_dataloader

In [ ]:
train_dataloader, validation_dataloader, test_dataloader = (
    create_dataloaders(train_dir=train_images, validation_dir=validation_images,test_dir=test_images, transform=model_transforms,batch_size=8))  # removel transforms=model_transforms_combine

In [ ]:
class DataGeneratorUsingAugmentation(Dataset):
    def __init__(self, dataset_path, mean_BC, std_BC, s=0.5):
        self.dataset_path = dataset_path
        self.mean_BC = mean_BC
        self.std_BC = std_BC
        self.images = (glob.glob(os.path.join(self.dataset_path, "*.tif")) + glob.glob(os.path.join(self.dataset_path, "*.jpg")) +
                       glob.glob(os.path.join(self.dataset_path, "*.png")))
        self.s = s
        self.transforms = transforms.Compose([transforms.RandomHorizontalFlip(p=0.5),
             transforms.RandomResizedCrop(50, scale=(0.8, 1.0)),
             transforms.ColorJitter(0.4 * self.s, 0.4 * self.s, 0.4 * self.s, 0.1 * self.s),
             transforms.RandomGrayscale(p=0.2),
             transforms.GaussianBlur(kernel_size=3),
             transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.9, 1.1), shear=5),
             transforms.ToTensor(),
             transforms.Normalize(mean=self.mean_BC.tolist(), std=self.std_BC.tolist())])
        print(f'Found {len(self.images)} images in {self.dataset_path}')

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image_path = self.images[idx]
        image_name = os.path.splitext(image_path)[0].split('/')[-1]
        tissue_number = re.search(r'tissue(\d)', image_name)
        tissue_number = int(tissue_number.group(1))
        x = Image.open(image_path).convert("RGB")
        x1 = self.transforms(x)
        x2 = self.transforms(x)
        name_x1 = image_name + '_x1'
        name_x2 = image_name + '_x2'
        real_label_x1 = torch.tensor(tissue_number, dtype=torch.long)
        real_label_x2 = real_label_x1
        return x1, real_label_x1, name_x1, x2, real_label_x2, name_x2
    def on_epoch_end(self):
        self.images = random.sample(self.images, len(self.images))

In [ ]:
class DataGeneratorUsingAugmentation_valid_test(Dataset):
    def __init__(self, dataset_path, mean_BC, std_BC, s=0.5):
        self.dataset_path = dataset_path
        self.mean_BC = mean_BC
        self.std_BC = std_BC
        self.images = (glob.glob(os.path.join(self.dataset_path, "*.tif")) + glob.glob(os.path.join(self.dataset_path, "*.jpg")) +
                       glob.glob(os.path.join(self.dataset_path, "*.png")))
        self.s = s
        self.transforms = transforms.Compose([
            transforms.Resize((50, 50)),
            transforms.ToTensor(),
            transforms.Normalize(mean=self.mean_BC.tolist(), std=self.std_BC.tolist()) ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image_path = self.images[idx]
        image_name = os.path.splitext(image_path)[0].split('/')[-1]
        tissue_number = re.search(r'tissue(\d)', image_name)
        tissue_number = int(tissue_number.group(1))
        x = Image.open(image_path).convert("RGB")
        x1 = self.transforms(x)
        x2 = self.transforms(x)
        name_x1 = image_name + '_x1'
        name_x2 = image_name + '_x2'
        real_label_x1 = torch.tensor(tissue_number, dtype=torch.long)

        real_label_x2 = real_label_x1
        return x1, real_label_x1, name_x1, x2, real_label_x2, name_x2
    def on_epoch_end(self):
        self.images = random.sample(self.images, len(self.images))

In [ ]:
class DataGenerator(Dataset):
    def __init__(self, dataset_path, mean_BC, std_BC, s=0.5):
        self.dataset_path = dataset_path
        self.mean_BC = mean_BC
        self.std_BC = std_BC
        self.images = (glob.glob(os.path.join(self.dataset_path, "*.tif")) + glob.glob(
            os.path.join(self.dataset_path, "*.jpg"))+ glob.glob(os.path.join(self.dataset_path, "*.png")))
        self.s = s
        self.transforms = transforms.Compose([
            transforms.Resize((50, 50)),
            transforms.ToTensor(),
            transforms.Normalize(mean=self.mean_BC.tolist(), std=self.std_BC.tolist()) ])
        print(f'Found {len(self.images)} images in {self.dataset_path}')

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image_path = self.images[idx]
        image_name = os.path.splitext(image_path)[0].split('/')[-1]
        tissue_number = re.search(r'tissue(\d)', image_name)
        tissue_number = int(tissue_number.group(1))
        x = Image.open(image_path).convert("RGB")
        x = self.transforms(x)
        name_x = image_name
        real_label_x = torch.tensor(tissue_number, dtype=torch.long)

        return x, real_label_x, name_x
    def on_epoch_end(self):
        self.images = random.sample(self.images, len(self.images))

In [ ]:
all_train_images_aug = DataGeneratorUsingAugmentation(train_images,mean_BC, std_BC)
dl_all_train_images_aug = DataLoader(all_train_images_aug, batch_size=8, drop_last=True)
#######
all_valid_images_aug = DataGeneratorUsingAugmentation_valid_test(validation_images,mean_BC, std_BC)
dl_all_valid_images_aug = DataLoader(all_valid_images_aug, batch_size=8, drop_last=True)
#########################
dg_actual_train = DataGenerator(train_images,mean_BC, std_BC)
dl_actual_train_images = DataLoader(dg_actual_train, batch_size=8, drop_last=True)
######
dg_validation = DataGenerator(validation_images,mean_BC, std_BC)
dl_validation = DataLoader(dg_validation, batch_size=8, drop_last=True)
######
dg_test = DataGenerator(test_images,mean_BC, std_BC)
dl_test = DataLoader(dg_test, batch_size=8, drop_last=True)

In [ ]:
# optimizer
EETA_DEFAULT = 0.001
class LARS(Optimizer):
    def __init__(self, params, lr=required, momentum=0.9, use_nesterov=False, weight_decay=0.0,
                 exclude_from_weight_decay=None, exclude_from_layer_adaptation=None, classic_momentum=True,eeta=EETA_DEFAULT, ):
        self.epoch = 0
        defaults = dict(lr=lr, momentum=momentum, use_nesterov=use_nesterov, weight_decay=weight_decay,
                        exclude_from_weight_decay=exclude_from_weight_decay,
                        exclude_from_layer_adaptation=exclude_from_layer_adaptation, classic_momentum=classic_momentum,
                        eeta=eeta, )
        super(LARS, self).__init__(params, defaults)
        self.lr = lr
        self.momentum = momentum
        self.weight_decay = weight_decay
        self.use_nesterov = use_nesterov
        self.classic_momentum = classic_momentum
        self.eeta = eeta
        self.exclude_from_weight_decay = exclude_from_weight_decay
        if exclude_from_layer_adaptation:
            self.exclude_from_layer_adaptation = exclude_from_layer_adaptation
        else:
            self.exclude_from_layer_adaptation = exclude_from_weight_decay
    def step(self, epoch=None, closure=None):
        loss = None
        if closure is not None:
            loss = closure()
        if epoch is None:
            epoch = self.epoch
            self.epoch += 1
        for group in self.param_groups:
            weight_decay = group["weight_decay"]
            momentum = group["momentum"]
            eeta = group["eeta"]
            lr = group["lr"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                param = p.data
                grad = p.grad.data
                param_state = self.state[p]
                # TODO: get param names
                # if self._use_weight_decay(param_name):
                grad += self.weight_decay * param
                if self.classic_momentum:
                    trust_ratio = 1.0
                    # TODO: get param names
                    # if self._do_layer_adaptation(param_name):
                    w_norm = torch.norm(param)
                    g_norm = torch.norm(grad)
                    device = g_norm.get_device()
                    trust_ratio = torch.where(w_norm.gt(0), torch.where(g_norm.gt(0),(self.eeta * w_norm / g_norm),
                          torch.Tensor([1.0]).to(device), ),torch.Tensor([1.0]).to(device), ).item()
                    scaled_lr = lr * trust_ratio
                    if "momentum_buffer" not in param_state:
                        next_v = param_state["momentum_buffer"] = torch.zeros_like(p.data)
                    else:
                        next_v = param_state["momentum_buffer"]
                    next_v.mul_(momentum).add_(scaled_lr, grad)
                    if self.use_nesterov:
                        update = (self.momentum * next_v) + (scaled_lr * grad)
                    else:
                        update = next_v
                    p.data.add_(-update)
                else:
                    raise NotImplementedError
        return loss

    def _use_weight_decay(self, param_name):
        """Whether to use L2 weight decay for `param_name`."""
        if not self.weight_decay:
            return False
        if self.exclude_from_weight_decay:
            for r in self.exclude_from_weight_decay:
                if re.search(r, param_name) is not None:
                    return False
        return True
    def _do_layer_adaptation(self, param_name):
        """Whether to do layer-wise learning rate adaptation for `param_name`."""
        if self.exclude_from_layer_adaptation:
            for r in self.exclude_from_layer_adaptation:
                if re.search(r, param_name) is not None:
                    return False
        return True
###################################

In [ ]:
#model
class Identity(nn.Module):
    def __init__(self):
        super(Identity, self).__init__()
    def forward(self, x):
        return x
#######################################################
class LinearLayer(nn.Module):
    def __init__(self, in_features, out_features, use_bias=True, use_bn=False, **kwargs):
        super(LinearLayer, self).__init__(**kwargs)
        self.in_features = in_features
        self.out_features = out_features
        self.use_bias = use_bias
        self.use_bn = use_bn
        self.linear = nn.Linear(self.in_features, self.out_features, bias=self.use_bias and not self.use_bn)
        if self.use_bn:
            self.bn = nn.BatchNorm1d(self.out_features)
    def forward(self, x):
        x = self.linear(x)
        if self.use_bn:
            x = self.bn(x)
        return x
########################################################
class ProjectionHead(nn.Module):
    def __init__(self, in_features, hidden_features, out_features, head_type='nonlinear', **kwargs):
        super(ProjectionHead, self).__init__(**kwargs)
        self.in_features = in_features
        self.out_features = out_features
        self.hidden_features = hidden_features
        self.head_type = head_type
        if self.head_type == 'linear':
            self.layers = LinearLayer(self.in_features, self.out_features, False, True)
        elif self.head_type == 'nonlinear':
            self.layers = nn.Sequential(
                LinearLayer(self.in_features, self.hidden_features, True, True),
                nn.ReLU(),  # a nonlinear function
                LinearLayer(self.hidden_features, self.out_features, False, True))
    def forward(self, x):
        x = self.layers(x)
        return x
##########################################################
class CluSiam(nn.Module):
    def __init__(self, batch_size):
        super(CluSiam, self).__init__()
        self.pretrained = models.resnet50(pretrained=True)
        self.pretrained.conv1 = nn.Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), bias=False)
        # self.pretrained.layer_norm = nn.LayerNorm(64)  # add batchnormalization
        self.pretrained.maxpool = Identity()
        self.pretrained.fc = Identity()
        self.batch_size = batch_size
        for p in self.pretrained.parameters():
            p.requires_grad = True
        self.projector = ProjectionHead(2048, 1000, 1000)
        self.classification_label_BC = nn.Sequential(nn.Linear(2048, 1024),
                                                     nn.BatchNorm1d(1024),
                                                     nn.ReLU(),
                                                     nn.Dropout(0.5),
                                                     nn.Linear(1024, 512),
                                                     nn.BatchNorm1d(512),
                                                     nn.ReLU(),
                                                     nn.Dropout(0.5),
                                                     nn.Linear(512, 256),
                                                     nn.BatchNorm1d(256),
                                                     nn.ReLU(),
                                                     nn.Dropout(0.5),
                                                     nn.Linear(256, 1))
        #####
    def forward(self, x, real_label_x, name_x):
        h = self.pretrained(x)
        z = self.projector(h)
        y_predict = self.classification_label_BC(h)
        return h, z, y_predict

In [ ]:
#loss_function
class loss_function(nn.Module):
    def __init__(self, batch_size, temperature, lambda1=0.5):
        super(loss_function, self).__init__()
        self.batch_size = batch_size
        self.temperature = temperature
        self.lambda1 = lambda1
        self.mask = self.mask_correlated_samples(batch_size)
        self.criterion = nn.CrossEntropyLoss(reduction="mean")  # Cross-Entropy Loss for similarity clr
        self.similarity_f = nn.CosineSimilarity(dim=2)
        self.classification_criterion = nn.BCEWithLogitsLoss()
        #self.classification_criterion = nn.CrossEntropyLoss()
    def mask_correlated_samples(self, batch_size):
        N = 2 * batch_size
        mask = torch.ones((N, N), dtype=bool)
        mask = mask.fill_diagonal_(0)
        for i in range(batch_size):
            mask[i, batch_size + i] = 0
            mask[batch_size + i, i] = 0
        return mask

    def forward(self, h1, z1, h2, z2, y_predict1, y_predict2, real_label_x1, real_label_x2):
        N = 2 * self.batch_size
        z = torch.cat((z1, z2), dim=0)
        sim_matrix = self.similarity_f(z.unsqueeze(1), z.unsqueeze(0)) / self.temperature
        sim1 = torch.diag(sim_matrix, self.batch_size)
        sim2 = torch.diag(sim_matrix, -self.batch_size)
        positive_samples = torch.cat((sim1, sim2), dim=0).reshape(N, 1)
        negative_samples = sim_matrix[self.mask].reshape(N, -1)
        labels = torch.from_numpy(np.array([0] * N)).reshape(-1).to(positive_samples.device).long()
        logits = torch.cat((positive_samples, negative_samples), dim=1)
        loss_clr = self.criterion(logits, labels)
        loss_clr /= N
        ####################
        loss_x1 = 0
        loss_x2 = 0
        loss_x1 += self.classification_criterion(y_predict1, real_label_x1)
        loss_x2 += self.classification_criterion(y_predict2, real_label_x2)
        loss_classifier_BC = loss_x1 + loss_x2
        #####################################
        l1_reg = torch.tensor(0., requires_grad=True).to(z.device)
        for param in model.parameters():
            l1_reg += torch.norm(param, 2)
        loss_classifier = loss_classifier_BC + self.lambda1 * l1_reg
        #####################
        total_loss = 0.3 * loss_clr + 0.7 * loss_classifier
        ##############
        return total_loss, loss_clr,loss_classifier, sim_matrix

In [ ]:
#call the model, call the optimizer, call the loss_function
print("CUDA available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
model = CluSiam(batch_size=8)
model = model.to('cuda:0')
###################################
optimizer = LARS([params for params in model.parameters() if params.requires_grad],
                 lr=0.001, weight_decay=1e-6, exclude_from_weight_decay=["batch_normalization", "bias"], )
#################################################
warmupscheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda epoch: (epoch + 1) / 10.0)  # , verbose = True)
mainscheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, 500, eta_min=0.05)  # , verbose = True)
loss_fn = loss_function(batch_size=8, temperature=0.5)
#############################################################

In [ ]:
#training and validation classes
def train_clr(dataloaderr, model, optimizer):
    total_loss_epoch = 0
    loss_classifier_epoch = 0
    loss_clr_epoch = 0
    for step, (x1, real_label_x1, name_x1, x2, real_label_x2, name_x2) in enumerate(dataloaderr):
        optimizer.zero_grad()
        x1 = x1.squeeze().to('cuda:0').float()
        x2 = x2.squeeze().to('cuda:0').float()
        real_label_x1 = real_label_x1.to('cuda:0')
        real_label_x2 = real_label_x2.to('cuda:0')
        real_label_x1 = real_label_x1.unsqueeze(1).float()
        real_label_x2 = real_label_x2.unsqueeze(1).float()

        h1, z1, y_predict1 = model(x1, real_label_x1, name_x1)
        h2, z2, y_predict2 = model(x2, real_label_x2, name_x2)
        total_loss, loss_clr,loss_classifier, sim_matrix = loss_fn(h1, z1, h2, z2, y_predict1, y_predict2, real_label_x1, real_label_x2 )
        if dataloaderr == dl_all_train_images_aug:
            total_loss.backward()
            optimizer.step()
        total_loss_epoch += total_loss.item()
        loss_clr_epoch += loss_clr.item()
        loss_classifier_epoch += loss_classifier.item()
    return total_loss_epoch, loss_clr_epoch,loss_classifier_epoch
########################################################################
def validation(dataloader, model):
    all_real_labels_BC = []
    all_preds_labels_BC = []
    batch_size = 8
    for (x, real_label_x, name_x) in dataloader:
        x = x.to('cuda:0').float()
        real_label_x = real_label_x.to('cuda:0')
        h, _, y_predict = model(x, real_label_x, name_x)
        preds = (torch.sigmoid(y_predict) > 0.5).long().squeeze()
        #preds = y_predict.argmax(dim=1)

        all_real_labels_BC.extend(real_label_x.cpu().numpy())
        all_preds_labels_BC.extend(preds.cpu().numpy())
    accuracy_BC = accuracy_score(all_real_labels_BC, all_preds_labels_BC)
    return accuracy_BC
#########################################################################

In [ ]:
#training the model
# start training
clr_loss_all_train = []
clr_loss_all_valid = []
classifier_loss_all_train = []
classifier_loss_all_valid = []
accuracy_actual_train_BC = []
accuracy_valid_BC = []
epochs = 5
current_epoch = 0
name = "model_epoch_{}.pth"
learning_rate_plot = []
best_accuracy = 0.0
step_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

for epoch in range(epochs):
    model.train()
    total_loss_epoch_train, loss_clr_epoch_train,classifier_loss_epoch_train = train_clr(dl_all_train_images_aug, model, optimizer)
    lr = optimizer.param_groups[0]["lr"]
    learning_rate_plot.append(lr)

    if epoch < 15:
        warmupscheduler.step()
    elif epoch >= 15 and epoch < 25:
        mainscheduler.step()
    elif epoch >= 25:
        step_scheduler.step()

    model.eval()
    with torch.no_grad():
        total_loss_epoch_valid, loss_clr_epoch_valid,classifier_loss_epoch_valid = train_clr(dl_all_valid_images_aug, model, optimizer)
        accuracy_train_BC = validation(dl_actual_train_images, model)
        accuracy_validation_BC = validation(dl_validation, model)

    clr_loss_all_train.append(total_loss_epoch_train / len(dl_all_train_images_aug))
    clr_loss_all_valid.append(total_loss_epoch_valid / len(dl_all_valid_images_aug))
    classifier_loss_all_train.append(classifier_loss_epoch_train / len(dl_all_train_images_aug))
    classifier_loss_all_valid.append(classifier_loss_epoch_valid / len(dl_all_valid_images_aug))
    accuracy_valid_BC.append(accuracy_validation_BC)
    ###################
    data = [["Dataset", "Training Accuracy", "Validation Accuracy"],
            ["BC", f"{accuracy_train_BC:.2f}", f"{accuracy_validation_BC:.2f}"]]
    from datetime import datetime
    current_time = datetime.now().strftime("%H:%M:%S")
    print(f"[{current_time}] Epoch [{epoch + 1}/{epochs}] - Loss CLR_train: {total_loss_epoch_train / len(dl_all_train_images_aug):.2f} | Loss CLR_valid: {total_loss_epoch_valid / len(dl_all_valid_images_aug):.2f}")
    print(tabulate(data, headers="firstrow", tablefmt="pretty"))
    #####################
    current_epoch += 1
    all_train_images_aug.on_epoch_end()

In [ ]:
# save_model_and_features(model, current_epoch, name, save_dir_train)
print("Training the model is completed and the model is saved in :", current_epoch)  # saved in the last epoch

In [ ]:
def plot_with_umap2(model, dataloader, filename):
    all_z_vectors_BC = []
    all_h_vectors_BC = []
    all_labels_real_for_cluster_BC = []
    batch_size = 8
    model.eval()
    with torch.no_grad():
        for step, (x, real_label_x, name_x) in enumerate(dataloader):
            x = x.squeeze().to('cuda:0').float()
            real_label_x = real_label_x.to('cuda:0')
            h, z, _ = model(x, real_label_x, name_x)
            for i in range(batch_size):
                all_z_vectors_BC.append(z[i].cpu().numpy())
                all_h_vectors_BC.append(h[i].cpu().numpy())
                all_labels_real_for_cluster_BC.append(real_label_x[i].cpu().numpy())

    all_z_vectors_BC = np.array(all_z_vectors_BC)
    all_h_vectors_BC = np.array(all_h_vectors_BC)
    all_labels_real_for_cluster_BC = np.array(all_labels_real_for_cluster_BC)

    # UMAP for BC
    reducer_BC = umap.UMAP(n_components=2, random_state=42, n_jobs=1)
    reduced_features_z_BC = reducer_BC.fit_transform(all_z_vectors_BC)
    reduced_features_h_BC = reducer_BC.fit_transform(all_h_vectors_BC)

    # Plot BC clusters
    label_colors_BC = {0: 'red', 1: 'blue'}
    colors_BC = [label_colors_BC[label] for label in all_labels_real_for_cluster_BC]
    handles_BC = [plt.Line2D([0], [0], color=label_colors_BC[i],markerfacecolor=label_colors_BC[i], markersize=10, label=f'{desc}')
    for i, desc in enumerate(['Non-tumor', 'Tumor'])]
    plt.figure(figsize=(6, 4))
    plt.scatter(reduced_features_z_BC[:, 0], reduced_features_z_BC[:, 1],c=colors_BC, alpha=0.5)
    plt.legend(handles=handles_BC, loc='best')
    plt.title(f'BC Cancer Clusters using UMAP')
    plt.savefig(f'{filename}_BC_z_umap.png')
    plt.show()
    plt.figure(figsize=(6, 4))
    plt.scatter(reduced_features_h_BC[:, 0], reduced_features_h_BC[:, 1],c=colors_BC, alpha=0.5)
    plt.legend(handles=handles_BC, loc='best')
    plt.title(f'BC Cancer Clusters using UMAP')
    plt.savefig(f'{filename}_BC_h_umap.png')
    plt.show()
    return all_z_vectors_BC, all_h_vectors_BC, all_labels_real_for_cluster_BC

In [ ]:
#plot the UMAP for tumor and non-tumor feature representations (the first cluster is representation the z vector, and the second map is represenation the h vectors )
all_z_vectors3, all_h_vectors3, all_labels_real_for_cluster3 = plot_with_umap2(model, dl_test,"goal_clustering_with_umap_test")  # UMAP

In [ ]:
#calculating the metrics
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
def test(model, dataloader, output_file='cleaning_testing_dataset.txt', mode='test'):
    model.eval()
    all_labels_real_BC = []
    all_labels_preds_BC = []
    all_probabilities_BC = []
    batch_size = 8
    high_conf_file = output_file.replace('.txt', '_high_confidence_training_dataset.txt')
    with open(output_file, 'w') as f, open(high_conf_file, 'w') as f_high_conf:
        f.write("PatchName\tRealLabel\tPredictedLabel\tConfidence\n")
        f_high_conf.write("PatchName\tRealLabel\tPredictedLabel\tConfidence\n")
        with torch.no_grad():
            for (x, real_label_x, name_x) in dataloader:
                x = x.squeeze().to('cuda:0').float()
                real_label_x = real_label_x.to('cuda:0')
                _, _, label_x_pred = model(x, real_label_x, name_x)
                for i in range(x.size(0)):
                    real = real_label_x[i].cpu().item()
                    name = name_x[i]
                    #probabilities_BC = F.softmax(label_x_pred[i], dim=0).cpu().numpy()
                    #all_probabilities_BC.append(probabilities_BC)
                    probability = torch.sigmoid(label_x_pred[i]).cpu().item()
                    pred = int(probability > 0.5)  # 0 or 1
                    all_probabilities_BC.append([1 - probability, probability])
                    if pred == 1:
                        confidence = probability
                    else:
                        confidence = 1 - probability
                    f.write(f"{name}\t{real}\t{pred}\t{confidence:.4f}\n")
                    if real == pred and confidence > 0.9:
                        f_high_conf.write(f"{name}\t{real}\t{pred}\t{confidence:.4f}\n")
                    all_labels_real_BC.append(real)
                    all_labels_preds_BC.append(pred)

    # Evaluation Metrics (no change here)
    all_probabilities_BC = np.array(all_probabilities_BC)
    accuracy_BC = accuracy_score(all_labels_real_BC, all_labels_preds_BC)
    f1_BC = f1_score(all_labels_real_BC, all_labels_preds_BC, average='weighted', zero_division=0)
    num_classes_BC = 2
    all_labels_real_BC = np.array(all_labels_real_BC).astype(int)
    all_labels_real_one_hot_BC = np.eye(num_classes_BC)[all_labels_real_BC]
    auc_BC = roc_auc_score(all_labels_real_one_hot_BC, all_probabilities_BC, multi_class='ovr', average='macro')
    fpr_BC = {}
    tpr_BC = {}
    for i in range(num_classes_BC):
        fpr_BC[i], tpr_BC[i], _ = roc_curve(all_labels_real_one_hot_BC[:, i], all_probabilities_BC[:, i])

    class_accuracies_BC = {}
    class_precisions_BC = {}
    class_recalls_BC = {}
    class_f1_scores_BC = {}
    for i in range(num_classes_BC):
        class_labels_BC = np.array(all_labels_real_BC) == i
        class_preds_BC = np.array(all_labels_preds_BC) == i
        class_accuracies_BC[i] = accuracy_score(class_labels_BC, class_preds_BC)
        class_precisions_BC[i] = precision_score(class_labels_BC, class_preds_BC, zero_division=0)
        class_recalls_BC[i] = recall_score(class_labels_BC, class_preds_BC, zero_division=0)
        class_f1_scores_BC[i] = f1_score(class_labels_BC, class_preds_BC, zero_division=0)
    cm = confusion_matrix(all_labels_real_BC, all_labels_preds_BC)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Not Tumor', 'Tumor'])
    disp.plot(cmap=plt.cm.Blues)
    plt.title(f'Confusion Matrix')
    plt.savefig(f'confusion_matrix_{mode}.png')
    plt.close()
    return (accuracy_BC, auc_BC, fpr_BC, tpr_BC, f1_BC, class_accuracies_BC,class_precisions_BC, class_recalls_BC, class_f1_scores_BC)
#####################################################################
# evaluate model on test dataset
(accuracy_BC, auc_BC, fpr_BC, tpr_BC, f1_BC, class_accuracies_BC,class_precisions_BC, class_recalls_BC, class_f1_scores_BC) = (
    test(model,dl_test,output_file='cleaning_test_dataset.txt',mode='test'))

In [ ]:
# Plot ROC curve for BC
plt.figure()
for i in range(len(fpr_BC)):
    plt.plot(fpr_BC[i], tpr_BC[i], lw=2, label=f'{class_names_BC[i]}')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive')
plt.ylabel('True Positive')
plt.title('ROC Curve for BC cancer')
plt.legend(loc='best')
plt.savefig('ROC for BC_test')
plt.show()